# Used Car Price Prediction

This project compares a median-price baseline, Ridge regression, and CatBoost for estimating used car prices from vehicle characteristics. It also provides a command-line tool for generating predictions from a CSV file.

**Dataset:** [Used Cars on Hugging Face](https://huggingface.co/datasets/imbalet/used_cars), with 432,227 rows and 21 columns. Download `dataset.csv` and save it next to this notebook as `cars.csv`. The dataset covers the Russian market; some categorical values, such as region names, are in Russian.

**Target:** `price`, measured in Russian rubles (RUB). **Evaluation metric:** mean absolute error (MAE), also in RUB; lower is better.

The data is randomly split into training, validation, and test sets in a 60/20/20 ratio with a fixed random seed. Models are fitted on the training set, CatBoost's best iteration is selected using validation MAE, and the test set is reserved for final evaluation.

Requirements: Python, NumPy, pandas, scikit-learn, and CatBoost. Run the cells sequentially from top to bottom. The notebook saves the model, feature metadata, and a small CSV file for testing the prediction command.

## 1. Imports and Data Loading

In [34]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [35]:
df = pd.read_csv('cars.csv')
df.head()

,owners,year,price,region,mileage,mark,model,complectation,steering_wheel,gear_type,...,transmission,power,displacement,characteristics,color,body_type_type,body_type_name,super_gen_name,super_gen_year_from,super_gen_year_to
0,1,2021,1460000,Приморский край,56000,Nissan,Note,NaN,RIGHT,ALL_WHEEL_DRIVE,...,AUTOMATIC,82,1198.0,Autech Crossover 1.2hyb AT (82 л.с.) 4WD,040001,ALLROAD_5_DOORS,Внедорожник 5 дв. Autech Crossover,III,2020,2023.0
1,1,2011,1460000,Орловская область,119000,Skoda,Yeti,Elegance,LEFT,ALL_WHEEL_DRIVE,...,ROBOT,152,1798.0,1.8 AMT (152 л.с.) 4WD,97948F,ALLROAD_5_DOORS,Внедорожник 5 дв.,I,2009,2014.0
2,2,2019,1460000,Калужская область,73100,Kia,Rio,NaN,LEFT,FORWARD_CONTROL,...,AUTOMATIC,123,1591.0,1.6 AT (123 л.с.),200204,SEDAN,Седан,IV,2017,2020.0
3,2,2018,1459000,Москва и Московская область,103500,Volkswagen,Polo,Drive,LEFT,FORWARD_CONTROL,...,ROBOT,125,1395.0,1.4 AMT (125 л.с.),660099,SEDAN,Седан,V Рестайлинг,2014,2020.0
4,1,2022,1457000,Белгородская область,33641,CHERY,Tiggo 4 Pro,Action,LEFT,FORWARD_CONTROL,...,MECHANICAL,113,1499.0,1.5 MT (113 л.с.),FAFBFB,ALLROAD_5_DOORS,Внедорожник 5 дв.,I,2020,2024.0


## 2. Data Quality Checks

Inspect missing values, exact duplicate rows, column types, and numerical feature ranges.

In [36]:
missing = df.isna().sum().to_frame('count')
missing['percent'] = df.isna().mean().mul(100).round(2)
missing

,count,percent
owners,0,0.00
year,0,0.00
price,0,0.00
region,0,0.00
mileage,0,0.00
mark,0,0.00
model,0,0.00
complectation,260331,60.23
steering_wheel,0,0.00
gear_type,0,0.00


In [37]:
missing[missing["count"] > 0].sort_values("percent", ascending=False)

,count,percent
complectation,260331,60.23
super_gen_name,44115,10.21
super_gen_year_to,10240,2.37
displacement,2757,0.64


In [38]:
df.duplicated().sum()

np.int64(0)

In [39]:
df.info()
df.describe().T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 432227 entries, 0 to 432226
Data columns (total 21 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   owners               432227 non-null  int64  
 1   year                 432227 non-null  int64  
 2   price                432227 non-null  int64  
 3   region               432227 non-null  object 
 4   mileage              432227 non-null  int64  
 5   mark                 432227 non-null  object 
 6   model                432227 non-null  object 
 7   complectation        171896 non-null  object 
 8   steering_wheel       432227 non-null  object 
 9   gear_type            432227 non-null  object 
 10  engine               432227 non-null  object 
 11  transmission         432227 non-null  object 
 12  power                432227 non-null  int64  
 13  displacement         429470 non-null  float64
 14  characteristics      432227 non-null  object 
 15  color            

,count,mean,std,min,25%,50%,75%,max
owners,432227.0,2.460154e+00,1.188675e+00,1.0,1.0,2.0,4.0,4.0
year,432227.0,2.012029e+03,7.848552e+00,1923.0,2008.0,2013.0,2018.0,2024.0
price,432227.0,1.619113e+06,1.598853e+06,20000.0,565000.0,1118800.0,2080000.0,10000900.0
mileage,432227.0,1.620468e+05,1.027241e+05,1.0,85000.0,154123.0,223000.0,1000000.0
power,432227.0,1.520590e+02,7.805696e+01,10.0,102.0,132.0,180.0,1265.0
displacement,429470.0,1.974645e+03,7.362305e+02,217.0,1591.0,1690.0,1999.0,8285.0
super_gen_year_from,432227.0,2.009057e+03,9.308468e+00,1923.0,2005.0,2011.0,2016.0,2024.0
super_gen_year_to,421987.0,2.014772e+03,7.091120e+00,1927.0,2011.0,2015.0,2020.0,2024.0


### Data Quality Findings

No exact duplicate rows were found. Missing values occur in trim level (60.23%), generation name (10.21%), generation end year (2.37%), and engine displacement (0.64%). The target price has no missing values.

Rows with missing values are retained. For Ridge, numerical missing values are filled with training-set medians, while categorical missing values are replaced with `unknown`. CatBoost uses the same categorical preparation and handles numerical missing values internally.

Checking exact duplicate rows does not rule out repeated listings with changed attributes. The absence of listing dates also limits our ability to evaluate performance on future data.

## 3. Train, Validation, and Test Split

Separate the target price from the input features. All learned preprocessing is fitted after splitting and uses only the training set.

In [41]:
X = df.drop(columns='price')
y = df['price']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

## 4. Baseline: Median Price

Predict the training-set median price for every vehicle. This provides a reference for measuring the benefit of the trained models.

In [42]:
median_price = y_train.median()
baseline_pred = np.full(len(y_valid), median_price)

baseline_mae = mean_absolute_error(y_valid, baseline_pred)

print('median price: ', median_price)
print('MAE baseline: ', baseline_mae)

median price:  1113500.0
MAE baseline:  1049181.698848979


## 5. Ridge Regression

Impute numerical features with training-set medians and standardize them. Replace missing categorical values with `unknown` and apply one-hot encoding. Combine preprocessing and regression in a single pipeline.

In [44]:
numeric_cols = X_train.select_dtypes(include='number').columns.tolist()
categorical_cols = X_train.select_dtypes(exclude='number').columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(
        strategy='constant',
        fill_value='unknown'
    )),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_cols),
    ('categorical', categorical_pipeline, categorical_cols)
])

ridge = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=1.0, solver='lsqr'))
])

ridge.fit(X_train, y_train)

ridge_pred = ridge.predict(X_valid)
ridge_mae = mean_absolute_error(y_valid, ridge_pred)

print(f'MAE baseline: {baseline_mae:,.0f}')
print(f'MAE Ridge:    {ridge_mae:,.0f}')

MAE baseline: 1,049,182
MAE Ridge:    228,880


### Ridge Results

Validation MAE is RUB 228,880, compared with RUB 1,049,182 for the median baseline: a reduction of approximately 78.2%. Ridge uses an initial configuration without hyperparameter tuning.

## 6. CatBoost

Pass categorical features directly, without one-hot encoding. Train for up to 1,000 iterations with `learning_rate=0.5` and select the best iteration using validation MAE. The training loss is RMSE; Ridge also uses a squared-error objective with regularization.

In [47]:
X_train_cb = X_train.copy()
X_valid_cb = X_valid.copy()

for col in categorical_cols:
    X_train_cb[col] = X_train_cb[col].fillna('unknown').astype(str)
    X_valid_cb[col] = X_valid_cb[col].fillna('unknown').astype(str)

catboost = CatBoostRegressor(
    iterations = 1000,
    depth = 6,
    learning_rate = 0.5,
    loss_function = 'RMSE',
    eval_metric = 'MAE',
    random_seed = 42,
    verbose = 100,
    allow_writing_files = False
)

catboost.fit(
    X_train_cb,
    y_train,
    cat_features=categorical_cols,
    eval_set=(X_valid_cb, y_valid),
    early_stopping_rounds=100
)

catboost_pred = catboost.predict(X_valid_cb)
catboost_mae = mean_absolute_error(y_valid, catboost_pred)

print(f"MAE baseline: {baseline_mae:,.0f}")
print(f"MAE Ridge:    {ridge_mae:,.0f}")
print(f"MAE CatBoost: {catboost_mae:,.0f}")

0:	learn: 695749.4122949	test: 696854.4061988	best: 696854.4061988 (0)	total: 134ms	remaining: 2m 13s
100:	learn: 211049.6933844	test: 211766.5997687	best: 211766.5997687 (100)	total: 6.04s	remaining: 53.8s
200:	learn: 195090.7032983	test: 198672.3599889	best: 198672.3599889 (200)	total: 12.2s	remaining: 48.4s
300:	learn: 187170.1278546	test: 192957.5640669	best: 192957.5640669 (300)	total: 18.5s	remaining: 42.9s
400:	learn: 181713.2744065	test: 189784.6449822	best: 189777.6875903 (399)	total: 25s	remaining: 37.4s
500:	learn: 177736.6722044	test: 187490.1722043	best: 187490.1722043 (500)	total: 31.7s	remaining: 31.5s
600:	learn: 174485.9005622	test: 186068.6921598	best: 186066.0333323 (598)	total: 38.5s	remaining: 25.5s
700:	learn: 171652.4326679	test: 185014.7576359	best: 185014.7576359 (700)	total: 45.4s	remaining: 19.4s
800:	learn: 169363.0685564	test: 184140.1034674	best: 184140.1034674 (800)	total: 52.7s	remaining: 13.1s
900:	learn: 167215.6625762	test: 183293.6514193	best: 183290

### CatBoost Results

Validation MAE is RUB 182,790, approximately 20.1% lower than Ridge. The best result was reached at the final, 1,000th iteration, so early stopping was not triggered.

## 7. Errors by Price Segment

Define three price segments using training-set price tertiles. Use the same boundaries for all models and compare MAE on the validation set.

In [50]:
q_low, q_high = y_train.quantile([1/3, 2/3])

errors = pd.DataFrame({
    'price': y_valid.to_numpy(),
    'Baseline': np.abs(y_valid.to_numpy() - median_price),
    'Ridge': np.abs(y_valid.to_numpy() - ridge_pred),
    'CatBoost': np.abs(y_valid.to_numpy() - catboost_pred)
})

errors['segment'] = pd.cut(
    errors['price'],
    bins = [-np.inf, q_low, q_high, np.inf],
    labels=['Low-price', 'Mid-price', 'High-price'],
)

print(f"Price segment boundaries: {q_low:,.0f} and {q_high:,.0f} RUB")

errors.groupby('segment', observed=True).agg(
    count=('price', 'size'),
    baseline_mae=('Baseline', 'mean'),
    ridge_mae=("Ridge", "mean"),
    catboost_mae=("CatBoost", "mean"),
).round(0)

Price segment boundaries: 739,500 and 1,676,000 RUB


,count,baseline_mae,ridge_mae,catboost_mae
segment,,,,
Low-price,28837,708497.0,139269.0,82843.0
Mid-price,28641,232227.0,152258.0,135881.0
High-price,28967,2196098.0,393848.0,328669.0


### Findings by Price Segment

The segment boundaries are RUB 739,500 and RUB 1,676,000. CatBoost achieves lower MAE in all three groups:

| Segment | Ridge MAE (RUB) | CatBoost MAE (RUB) |
|---|---:|---:|
| Low-price | 139,269 | 82,843 |
| Mid-price | 152,258 | 135,881 |
| High-price | 393,848 | 328,669 |

The largest percentage reduction in MAE relative to Ridge occurs in the low-price segment. Absolute errors are larger for expensive vehicles; assessing relative prediction errors would require a separate calculation.

## 8. Final Test Evaluation

Freeze the model configurations and evaluate all three approaches on the held-out test set. Test results are not used for hyperparameter selection.

In [51]:
X_test_cb = X_test.copy()

for col in categorical_cols:
    X_test_cb[col] = X_test_cb[col].fillna('unknown').astype(str)

test_predictions = {
    'baseline': np.full(len(y_test), median_price),
    'ridge': ridge.predict(X_test),
    'CatBoost': catboost.predict(X_test_cb)
}

test_results = pd.DataFrame([
    {
        'model': name,
        'test_mae': mean_absolute_error(y_test, pred)
    }
    for name, pred in test_predictions.items()
])

test_results.round(0)

,model,test_mae
0,baseline,1047220.0
1,ridge,230988.0
2,CatBoost,183696.0


### Conclusions

| Model | Validation MAE (RUB) | Test MAE (RUB) |
|---|---:|---:|
| Median baseline | 1,049,182 | 1,047,220 |
| Ridge | 228,880 | 230,988 |
| CatBoost | 182,790 | 183,696 |

CatBoost achieves the lowest test MAE in this experiment: RUB 183,696, approximately 20.5% lower than Ridge. It is selected for the command-line prediction tool.

The close validation and test results support performance on this particular random split. They do not establish performance on future listings or current market prices: the collection date is not sufficiently documented, no temporal split was performed, and duplicate checks were limited to exact row matches.

This experiment compares specific configurations rather than proving that one algorithm is always superior. No additional hyperparameter tuning was performed.

## 9. Save the Model and a Sample Input CSV

Save the fitted CatBoost model and its feature metadata. The prediction script uses the same feature order and categorical preprocessing.

In [54]:
model_dir = Path('models')
model_dir.mkdir(exist_ok=True)

catboost.save_model(str(model_dir / 'catboost.cbm'))

metadata = {
    'features' : X_train.columns.tolist(),
    'categorical_features' : list(categorical_cols)
}

with open(model_dir / 'metadata.json', 'w', encoding='utf-8') as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

In [56]:
X_test.head(10).to_csv("cars_example.csv", index=False)

## 10. Run CSV Predictions

After running the notebook, execute the following command in a terminal from the project directory:

```bash
python predict.py --input cars_example.csv --output predictions.csv
```

The script adds a `predicted_price` column to the original data. The input CSV must contain all training features; the target column `price` is not required.

To check that the script reproduces the notebook predictions, run the following in a separate cell after executing the command:

```python
result = pd.read_csv("predictions.csv")
np.testing.assert_allclose(
    result["predicted_price"],
    catboost.predict(X_test_cb.head(10)),
)
print("Script and notebook predictions match")
```

This check depends on a file created by an external command, so it is not part of the notebook's required execution sequence.